In [22]:
!pip install -U pip transformers

In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [24]:
checkpoint = 'facebook/nllb-200-distilled-600M'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [25]:
print(f"{len(tokenizer.vocab)}\n")

tokenizer.vocab

256204



{'bedingungen': 207484,
 '▁kanki': 139455,
 '▁ਪ੍ਰਗ': 178103,
 '▁Jesusasti': 72428,
 'ಕಟ್ಟ': 135387,
 'ァ': 250297,
 '▁فقر': 240603,
 '▁बस': 13605,
 'ciy': 221220,
 '▁ប្រសិន': 74194,
 '▁Untuk': 24170,
 'umų': 168580,
 '▁êm': 117933,
 'ýšľ': 189921,
 '▁الت': 6284,
 '▁гурав': 246968,
 '▁体': 225163,
 '▁civiliza': 151964,
 '▁wenyi': 102589,
 '▁hula': 129945,
 '▁артты': 48707,
 'ugata': 168523,
 '▁நப': 204296,
 'asitiri': 240684,
 'aldi': 88242,
 'ẻ': 250253,
 '▁देख': 4274,
 'हां': 12837,
 'ାଜିକ': 196743,
 '▁ⵉⴳ': 111414,
 '▁jangan': 26816,
 '▁ક્ર': 68066,
 '▁کشاور': 173815,
 '▁ማግኘት': 103590,
 '▁noqda': 217337,
 '▁پەیوەندیی': 218010,
 '▁ubio': 121551,
 '▁Balo': 214095,
 'Amb': 90589,
 '▁મિલ': 117845,
 'zab': 8494,
 '▁ଇସ': 237047,
 '▁Dona': 225687,
 '▁lgan': 50054,
 '▁mingħand': 196021,
 '▁తొలి': 208576,
 '▁काही': 15595,
 'ਝ': 250324,
 'હિ': 70683,
 'bilität': 187351,
 'ចារ្យ': 124778,
 '▁Öğ': 167661,
 '▁suhbat': 179841,
 'qaddas': 63363,
 '▁எங்களுக்கு': 139186,
 'ំពេញ': 184420,
 '▁rating': 127

In [26]:
thai_char_min = 0x0E00
thai_char_max = 0x0E7F

thai_tokens = [
    token for token in tokenizer.vocab.keys()
    if any(thai_char_min <= ord(char) <= thai_char_max for char in token)
]

thai_token_count = len(thai_tokens)
sample_size = 20
thai_tokens_sample = thai_tokens[:sample_size]


print(f"{thai_token_count}\n")
for token in thai_tokens_sample:
  print(token)

1712

▁ระ
ูก
รวจ
ตก
คุม
เดินทาง
ีก
ดื่ม
▁คือ
ํานา
ัตร
แม่
▁หน้า
บอกว่า
อบครัว
▁ควร
ยุ
โค
เหนือ
รม


In [27]:
import tensorflow as tf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import math

In [28]:
sentence = 'Work hard, play harder'

In [29]:
cleaned_sentence = sentence.replace(',', '')
cleaned_sentence

'Work hard play harder'

In [30]:
words = cleaned_sentence.split()
words

['Work', 'hard', 'play', 'harder']

In [32]:
sorted_words = sorted(words)
sorted_words

['Work', 'hard', 'harder', 'play']

In [33]:
dc = {word: index for index, word in enumerate(sorted_words)}
dc

{'Work': 0, 'hard': 1, 'harder': 2, 'play': 3}

In [34]:
sentence_int = tf.constant(
    [dc[s] for s in sentence.replace(',', '').split()],
    dtype=tf.int32
)

In [35]:
print(sentence)
print(sentence_int)

Work hard, play harder
tf.Tensor([0 1 3 2], shape=(4,), dtype=int32)


In [36]:
# สร้าง embedding layer
tf.random.set_seed(123)
vocab_size = 50_000
embedding_dim = 2

embed = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)

In [37]:
embedded_sentence = embed(sentence_int)

In [38]:
embedded_sentence

<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[-0.01130716, -0.03739134],
       [-0.00645286,  0.03358663],
       [-0.02700778, -0.00529767],
       [-0.01208397,  0.03202006]], dtype=float32)>

In [39]:
tf.random.set_seed(123)
vocab_size = 50_000
embedding_dim = 2

dummy_input = tf.constant([0, 1, 2], dtype=tf.int32)

# Case 1 Default initializer (RandomUniform(-0.05, 0.05))
embed_default = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)
_ = embed_default(dummy_input) # เรียกใช้งาน layer เพื่อสร้าง weights
weights_default = embed_default.get_weights()[0].flatten()
weights_default.shape

(100000,)

In [40]:
# Case 2 GlorotUniform initializer
tf.random.set_seed(123)
embed_glorot = tf.keras.layers.Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    embeddings_initializer=tf.keras.initializers.GlorotUniform()
)
_ = embed_glorot(dummy_input) # เรียกใช้งาน layer เพื่อสร้าง weights
weights_glorot = embed_glorot.get_weights()[0].flatten()
weights_glorot.shape

(100000,)

In [41]:
fig = make_subplots(rows=1, cols=1)

fig.add_trace(go.Histogram(x=weights_default, nbinsx=50, name="Default Uniform [-0.05, 0.05]", opacity=0.6))
fig.add_trace(go.Histogram(x=weights_glorot, nbinsx=50, name="Glorot Uniform", opacity=0.6))

fig.update_layout(
    title_text='Embedding Layer Initialization Comparison',
    xaxis_title_text='Weight values',
    yaxis_title_text='Frequency',
    barmode='overlay',
    legend_orientation="h",
    legend_yanchor="bottom",
    legend_y=1.02,
    legend_xanchor="right",
    legend_x=1
)

fig.show()

print("Default initializer range ", weights_default.min(), weights_default.max())
print("Glorot initializer range ", weights_glorot.min(), weights_glorot.max())

Default initializer range  -0.049999535 0.049998987
Glorot initializer range  -0.010954222 0.01095388


In [42]:
def glorot_uniform_limits(fan_in, fan_out):
    limit = math.sqrt(6.0 / (fan_in + fan_out))
    a, b = -limit, limit
    return a, b

# ตัวอย่าง Embedding layer (vocab_size=50000, embedding_dim=2)
fan_in = 50000
fan_out = 2

a, b = glorot_uniform_limits(fan_in, fan_out)
print("Glorot Uniform a =", a)
print("Glorot Uniform b =", b)

Glorot Uniform a = -0.010954232067652772
Glorot Uniform b = 0.010954232067652772


In [43]:
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

In [44]:
token_embedding_layer = model.model.encoder.embed_tokens
token_embedding_layer.weight.shape

torch.Size([256206, 1024])

In [45]:
long_sentence = "In the vast realm of natural language processing, understanding the nuances of how models handle sequential data is crucial. Positional encoding plays a vital role in providing this essential information to the model, allowing it to differentiate between words at different positions in a sentence, which is fundamental for tasks like translation, summarization, and text generation."

In [47]:
tokens = tokenizer(long_sentence, return_tensors="pt")

print(tokens['input_ids'][0])

tensor([256047,    717,    349,  14430,  12284, 248070,    452,  25307,  65445,
        157278, 248079, 133930,    349,    713,  75831,    452,  11657, 141057,
         47274, 116914, 124785,   6067,    248, 182071, 248075,  12013,  58409,
         12025, 246156,   3054,    705,      9, 104781,  76065,    108, 174693,
          3423, 140515,  18781,    202,    349,  14916, 248079,  82935,     87,
           796,    202,  53054,    502,  25914,  51744,    230,  30158, 199073,
           108,      9, 109267, 248079,   9089,    248,  75529,    351, 226047,
          6399, 200356, 248079,   2493, 109207, 181953, 248079,    540,  35883,
        120531, 248075,      2])


In [48]:
len(tokens['input_ids'][0])

75

In [49]:
token_embedding_layer(tokens['input_ids'][0][0]).shape

torch.Size([1024])

In [50]:
token_embeddings = token_embedding_layer(tokens['input_ids'][0])

print("Token Embedding Matrix shape", token_embeddings.shape)
token_embeddings

Token Embedding Matrix shape torch.Size([75, 1024])


tensor([[-5.0000e+00, -1.2725e+00, -9.3604e-01,  ..., -1.8297e+01,
         -9.1328e+00, -1.0672e+01],
        [ 2.6416e-01,  2.6831e-01,  2.0117e-01,  ...,  3.2715e+00,
         -3.2402e+00,  3.1738e+00],
        [ 4.3579e-01, -2.3352e-01,  2.6825e-02,  ...,  5.4648e+00,
          2.7129e+00,  5.5430e+00],
        ...,
        [ 8.5859e+00, -4.5391e+00, -4.7314e-01,  ..., -7.9529e-02,
          7.4844e+00, -7.5156e+00],
        [-2.4863e+00, -2.7515e-01,  5.6114e-03,  ...,  1.0180e+01,
         -7.2422e+00, -4.8047e+00],
        [-7.8320e-01, -9.0527e-01, -9.4482e-01,  ...,  3.1078e+01,
         -8.1494e-01, -8.7354e-01]], grad_fn=<MulBackward0>)

In [51]:
import plotly.express as px

token_embeddings_np = token_embeddings.detach().numpy()

fig = px.imshow(
    token_embeddings_np,
    color_continuous_scale="RdBu",
    labels=dict(x="Embedding Dimension", y="Token Index", color="Value"),
    title="Token Embedding Heatmap"
)

fig.update_xaxes(side="top")
fig.update_layout(height=500, width=900)
fig.show()

In [52]:
d = embedded_sentence.shape[-1]
d

2

In [53]:
d_q, d_k, d_v = 2, 2, 4

d_q, d_k, d_v

(2, 2, 4)

In [54]:
tf.random.set_seed(123)
W_query = tf.Variable(tf.random.uniform((d, d_q)), trainable=True)
W_key   = tf.Variable(tf.random.uniform((d, d_k)), trainable=True)
W_value = tf.Variable(tf.random.uniform((d, d_v)), trainable=True)

In [55]:
print(W_query.shape, W_key.shape, W_value.shape)

(2, 2) (2, 2) (2, 4)


In [56]:
W_query

<tf.Variable 'Variable:0' shape=(2, 2) dtype=float32, numpy=
array([[0.12615311, 0.5727513 ],
       [0.2993133 , 0.5461836 ]], dtype=float32)>

In [57]:
W_key

<tf.Variable 'Variable:0' shape=(2, 2) dtype=float32, numpy=
array([[0.88968754, 0.12354946],
       [0.7718717 , 0.6850728 ]], dtype=float32)>

In [58]:
W_value

<tf.Variable 'Variable:0' shape=(2, 4) dtype=float32, numpy=
array([[0.48962688, 0.5857923 , 0.36451697, 0.6550509 ],
       [0.9075084 , 0.37557673, 0.6882372 , 0.25384045]], dtype=float32)>

In [59]:
embedded_sentence

<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[-0.01130716, -0.03739134],
       [-0.00645286,  0.03358663],
       [-0.02700778, -0.00529767],
       [-0.01208397,  0.03202006]], dtype=float32)>

In [60]:
queries = tf.matmul(embedded_sentence, W_query)
keys    = tf.matmul(embedded_sentence, W_key)
values  = tf.matmul(embedded_sentence, W_value)

In [61]:
print("Queries shape", queries.shape)
queries

Queries shape (4, 2)


<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[-0.01261816, -0.02689873],
       [ 0.00923888,  0.01464858],
       [-0.00499278, -0.01836224],
       [ 0.0080596 ,  0.01056772]], dtype=float32)>

In [62]:
print("Keys shape", keys.shape)
keys

Keys shape (4, 2)


<tf.Tensor: shape=(4, 2), dtype=float32, numpy=
array([[-0.03892116, -0.02701278],
       [ 0.02018354,  0.02221204],
       [-0.02811761, -0.00696609],
       [ 0.01396442,  0.0204431 ]], dtype=float32)>

In [63]:
print("Values shape", values.shape)
values

Values shape (4, 4)


<tf.Tensor: shape=(4, 4), dtype=float32, numpy=
array([[-0.03946925, -0.02066696, -0.02985576, -0.0168982 ],
       [ 0.02732065,  0.00883432,  0.02076339,  0.00429869],
       [-0.01803142, -0.01781064, -0.01349085, -0.01903624],
       [ 0.02314183,  0.00494729,  0.01763258,  0.00021237]],
      dtype=float32)>

In [64]:
omega = tf.matmul(queries, keys, transpose_b=True)

print("Omega shape", omega.shape)
print("Omega (Unnormalized attention weights)")
print(omega)

Omega shape (4, 4)
Omega (Unnormalized attention weights)
tf.Tensor(
[[ 0.00121772 -0.00085215  0.00054217 -0.0007261 ]
 [-0.00075529  0.00051185 -0.00036182  0.00042848]
 [ 0.00069034 -0.00050863  0.0002683  -0.0004451 ]
 [-0.00059915  0.0003974  -0.00030023  0.00032858]], shape=(4, 4), dtype=float32)


In [65]:
d_k = tf.cast(d_k, tf.float32)

scaled_omega = omega / tf.sqrt(d_k)

attention_weights = tf.nn.softmax(scaled_omega, axis=-1)

print("Attention Weights")
print(attention_weights)


Attention Weights
tf.Tensor(
[[0.25020728 0.24984133 0.25008777 0.24986358]
 [0.24987431 0.2500983  0.24994384 0.25008357]
 [0.25012183 0.24990986 0.2500472  0.24992108]
 [0.24990176 0.2500779  0.24995458 0.25006574]], shape=(4, 4), dtype=float32)


In [66]:
row_sums = tf.reduce_sum(attention_weights, axis=-1)

print("Sum of each row in attention_weights")
row_sums

Sum of each row in attention_weights


<tf.Tensor: shape=(4,), dtype=float32, numpy=array([1., 1., 1., 1.], dtype=float32)>

In [67]:
context_vector = tf.matmul(attention_weights, values)

print("Context Vector shape", context_vector.shape)
print(context_vector)

Context Vector shape (4, 4)
tf.Tensor(
[[-0.0017768  -0.00618192 -0.00125073 -0.00786173]
 [-0.00174895 -0.00616912 -0.00122964 -0.00785221]
 [-0.00176949 -0.00617854 -0.0012452  -0.0078592 ]
 [-0.0017512  -0.00617014 -0.00123134 -0.00785297]], shape=(4, 4), dtype=float32)


In [68]:
class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, d_in, d_out_kq, d_out_v):
        super().__init__()
        self.d_out_kq = d_out_kq

        self.W_query = tf.Variable(
            tf.random.uniform((d_in, d_out_kq)), trainable=True
        )
        self.W_key = tf.Variable(
            tf.random.uniform((d_in, d_out_kq)), trainable=True
        )
        self.W_value = tf.Variable(
            tf.random.uniform((d_in, d_out_v)), trainable=True
        )

    def call(self, x):
        keys = tf.matmul(x, self.W_key)      # [T, d_out_kq]
        queries = tf.matmul(x, self.W_query) # [T, d_out_kq]
        values = tf.matmul(x, self.W_value)  # [T, d_out_v]

        # Attention scores: QKᵀ
        attn_scores = tf.matmul(queries, keys, transpose_b=True)  # [T, T]

        # Softmax (scaled by sqrt(d_k))
        attn_weights = tf.nn.softmax(
            attn_scores / tf.math.sqrt(tf.cast(self.d_out_kq, tf.float32)), axis=-1
        )  # [T, T]

        # Weighted sum
        context_vec = tf.matmul(attn_weights, values)  # [T, d_out_v]
        return context_vec

In [69]:
tf.random.set_seed(123)

d_in, d_out_kq, d_out_v = 2, 2, 4

sa = SelfAttention(d_in, d_out_kq, d_out_v)

out = sa(embedded_sentence)

print(out.shape)  # (T, d_out_v)
print(out.numpy())

(4, 4)
[[-0.0017768  -0.00618192 -0.00125073 -0.00786173]
 [-0.00174895 -0.00616912 -0.00122964 -0.00785221]
 [-0.00176949 -0.00617854 -0.0012452  -0.0078592 ]
 [-0.0017512  -0.00617014 -0.00123134 -0.00785297]]


In [70]:
class MultiHeadAttentionWrapper(tf.keras.layers.Layer):
    def __init__(self, d_in, d_out_kq, d_out_v, num_heads):
        super().__init__()
        self.heads = [
            SelfAttention(d_in, d_out_kq, d_out_v)
            for _ in range(num_heads)
        ]

    def call(self, x):
        # รันทุก head แล้ว concat ตามแกนสุดท้าย
        head_outputs = [head(x) for head in self.heads]   # list of [T, d_out_v]
        return tf.concat(head_outputs, axis=-1)           # [T, num_heads * d_out_v]

In [71]:
tf.random.set_seed(123)

d_in, d_out_kq, d_out_v = 2, 2, 1

sa = SelfAttention(d_in, d_out_kq, d_out_v)

# ถ้า embedded_sentence.shape = [T, d_in] เช่น [6, 3]
out = sa(embedded_sentence)

print(out.shape)   # (T, d_out_v) -> (6, 1)
print(out.numpy())

(4, 1)
[[-0.00361428]
 [-0.00359573]
 [-0.00360941]
 [-0.00359722]]


In [72]:
tf.random.set_seed(123)

mha = MultiHeadAttentionWrapper(
    d_in, d_out_kq, d_out_v, num_heads=3
)

# run MHA
context_vecs = mha(embedded_sentence)   # [T, num_heads * d_out_v]

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tf.Tensor(
[[-0.00361428 -0.00259119 -0.00520924]
 [-0.00359573 -0.00258395 -0.0052    ]
 [-0.00360941 -0.00259001 -0.0052133 ]
 [-0.00359722 -0.00258476 -0.00520275]], shape=(4, 3), dtype=float32)
context_vecs.shape: (4, 3)
